In [1]:
!pip install influxdb-client pandas pyarrow huggingface_hub --quiet


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pandas as pd
from datetime import datetime
from influxdb_client import InfluxDBClient
from huggingface_hub import HfApi
from dotenv import load_dotenv

# --- 0. LOAD ENVIRONMENT VARIABLES ---
load_dotenv()

# --- 1. SECRETS & CONFIGURATION ---
INFLUX_URL = "https://us-east-1-1.aws.cloud2.influxdata.com"
INFLUX_TOKEN = os.getenv("INFLUX_TOKEN")
INFLUX_ORG = "Dewdu"
INFLUX_BUCKET = "sensor_data"

HF_TOKEN = os.getenv("HF_TOKEN")
HF_DATASET_REPO = "Dewdu/physiological-sensor-archive" 

# --- 2. QUERY INFLUXDB FOR THE LAST 7 DAYS ---
print("🕵️‍♀️ Querying InfluxDB for the last 7 days of data...")
client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG, timeout=30000)
query_api = client.query_api()

query = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -7d)
  |> filter(fn: (r) => r["_measurement"] == "physiological_metrics")
  |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
'''

tables = query_api.query(query)

# --- 3. CONVERT TO PANDAS DATAFRAME ---
print("🧹 Formatting data...")
records = []
for table in tables:
    for record in table.records:
        row = {
            "timestamp": record.get_time(),
            "user_id": record.values.get("user_id")
        }
        for i in range(10):
            row[f"f_{i}"] = record.values.get(f"f_{i}")
        records.append(row)

df = pd.DataFrame(records)

if df.empty:
    print("⚠️ No data found! Your bucket is completely empty because we just made it.")
else:
    df = df.sort_values(by=["user_id", "timestamp"]).reset_index(drop=True)
    print(f"✅ Found {len(df)} rows of sensor data.")

    # --- 4. SQUISH INTO A PARQUET FILE ---
    today_str = datetime.now().strftime("%Y_%m_%d")
    filename = f"sensor_archive_{today_str}.parquet"

    print(f"📦 Compressing into {filename}...")
    df.to_parquet(filename, engine='pyarrow', compression='snappy')

    # --- 5. PUSH TO HUGGING FACE DATASETS ---
    print("🚀 Uploading to Hugging Face Vault...")
    api = HfApi()

    api.upload_file(
        path_or_fileobj=filename,
        path_in_repo=f"data/{filename}", 
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        token=HF_TOKEN
    )

    print(f"🎉 Slay! Successfully archived {filename} to Hugging Face!")

c:\Users\H P\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🕵️‍♀️ Querying InfluxDB for the last 7 days of data...
🧹 Formatting data...
⚠️ No data found! Your bucket is completely empty because we just made it.
